## 1. Analytical Derivation: ScLERP to Constant Body Twist

Given a current pose $\boldsymbol{D}_1$ and a target pose $\boldsymbol{D}_2$ expressed as unit dual quaternions, the relative transformation required to reach the target pose is $\boldsymbol{D}_{12} = \boldsymbol{D}_1^{-1} \boldsymbol{D}_2$. 

The Spherical Linear Interpolation (ScLERP) trajectory parameterizes the path from $\boldsymbol{D}_1$ to $\boldsymbol{D}_2$ using the interpolation variable $s \in [0, 1]$:
$$\boldsymbol{D}(s) = \boldsymbol{D}_1 \boldsymbol{D}_{12}^s$$

Using the dual angle $\Phi = \theta + \epsilon d$ and the dual screw axis $\boldsymbol{L} = \hat{\boldsymbol{n}} + \epsilon \boldsymbol{m}$, the relative error can be expanded trigonometrically:
$$\boldsymbol{D}_{12}^s = \cos\left(\frac{s \Phi}{2}\right) + \boldsymbol{L} \sin\left(\frac{s \Phi}{2}\right)$$

To find the spatial body twist $\mathcal{V}_b$, we take the time derivative of the continuous trajectory. Applying the chain rule to the trigonometric expansion yields:
$$\dot{\boldsymbol{D}}(s) = \boldsymbol{D}_1 \left( -\frac{\dot{s} \Phi}{2} \sin\left(\frac{s \Phi}{2}\right) + \boldsymbol{L} \frac{\dot{s} \Phi}{2} \cos\left(\frac{s \Phi}{2}\right) \right)$$

The formal kinematic definition of the body twist is extracted by mapping the derivative back to the local frame:
$$\mathcal{V}_b = 2 \boldsymbol{D}^{-1}(s) \dot{\boldsymbol{D}}(s)$$

Substitute $\boldsymbol{D}(s)$ and $\dot{\boldsymbol{D}}(s)$ into the twist equation:
$$\mathcal{V}_b = 2 (\boldsymbol{D}_1 \boldsymbol{D}_{12}^s)^{-1} \left[ \boldsymbol{D}_1 \frac{d}{dt}(\boldsymbol{D}_{12}^s) \right]$$
$$\mathcal{V}_b = 2 (\boldsymbol{D}_{12}^s)^{-1} \boldsymbol{D}_1^{-1} \boldsymbol{D}_1 \frac{d}{dt}(\boldsymbol{D}_{12}^s)$$

Because $\boldsymbol{D}_1^{-1} \boldsymbol{D}_1 = 1$, the absolute starting pose cleanly cancels out. When the remaining trigonometric terms are multiplied against their inverse, they reduce entirely to the constant parameters of the screw axis:
$$\mathcal{V}_b = \dot{s} \Phi \boldsymbol{L}$$

Expanding the dual scalars and isolating the real (angular) and dual (linear) components:
$$\mathcal{V}_b = \dot{s} (\theta + \epsilon d)(\hat{\boldsymbol{n}} + \epsilon \boldsymbol{m})$$
$$\mathcal{V}_b = \dot{s} \left[ \theta \hat{\boldsymbol{n}} + \epsilon (\theta \boldsymbol{m} + d \hat{\boldsymbol{n}}) \right]$$

**Conclusion:** The body twist $\mathcal{V}_b$ is mathematically constant along the entire ScLERP trajectory. It is independent of the interpolation parameter $s$, meaning numerical time-stepping derivatives are unnecessary. 

---

## 2. Python Implementations

The following three methods compute the exact same 6D error twist $\mathcal{V} = [\boldsymbol{v}^T, \boldsymbol{\omega}^T]^T$ required for a proportional position-based servoing controller. All methods output the twist formatted as `[v_x, v_y, v_z, w_x, w_y, w_z]` to match standard Jacobian control architectures.

### Method A: Modern Robotics Matrix Logarithm
This is the standard $SE(3)$ matrix inversion and numerical matrix logarithm approach.


In [ ]:
import timeit
import numpy as np
import modern_robotics as mr
import pytransform3d.transformations as pt
import pytransform3d.rotations as pr

In [92]:

def p_servo_matrix(current_pose, target_pose, k, threshold):
    """Original method using Matrix Inversion and Matrix Logarithm."""
    if np.isscalar(k):
        k_mat = k * np.eye(6)
    else:
        k_mat = np.diag(k)
        
    bTd = np.linalg.inv(current_pose) @ target_pose
    ang_axis = mr.se3ToVec(mr.MatrixLog6(bTd))  #(omega, v)
    axis_ang = np.concatenate([ang_axis[3:], ang_axis[:3]])
    
    v = k_mat @ axis_ang #(v, omega)
    arrived = bool(np.sum(np.abs(axis_ang)) < threshold)
    return v, arrived




### Method B: Direct Dual Logarithm

This method maps the poses to dual quaternions and extracts the twist directly using the dual logarithm logic provided by `pytransform3d`.


In [93]:

def p_servo_dq(current_pose, target_pose, k, threshold):
    """Optimized method using Dual Quaternions."""
    if np.isscalar(k):
        k_mat = k * np.eye(6) 
    else:
        k_mat = np.diag(k)
        
    dq_c = current_pose
    dq_t = target_pose
    
    dq_c_inv = pt.dq_q_conj(dq_c)
    dq_e = pt.concatenate_dual_quaternions(dq_c_inv, dq_t)
    
    q, s_axis, h, theta = pt.screw_parameters_from_dual_quaternion(dq_e)
    
    if np.isinf(h):
        omega = np.zeros(3)
        v = theta * s_axis
    else:
        omega = theta * s_axis
        v = h * omega + np.cross(q, omega)
        
    axis_ang = np.concatenate((v, omega))
    v_cmd = k_mat @ axis_ang
    arrived = bool(np.sum(np.abs(axis_ang)) < threshold)
    return v_cmd, arrived



### Method C: ScLERP Analytical Derivative

This method implements the invariant $\dot{s} \Phi \boldsymbol{L}$ formulation proven in the analytical derivation. Because $s$ cancels out mathematically, it calculates the perfectly stable twist natively.



In [ ]:

from numba import njit

@njit
def p_servo_dq_fast(dq_c: np.ndarray, dq_t: np.ndarray, k_diag: np.ndarray, threshold: float):
    # 1. Conjugate of current pose (inverse)
    dq_c_inv = np.empty(8)
    dq_c_inv[0] = dq_c[0]
    dq_c_inv[1:4] = -dq_c[1:4]
    dq_c_inv[4] = dq_c[4]
    dq_c_inv[5:8] = -dq_c[5:8]
    
    # 2. Dual Quaternion Multiplication (dq_e = dq_c_inv * dq_t)
    dq_e = np.empty(8)
    # Real part multiplication
    dq_e[0] = dq_c_inv[0]*dq_t[0] - np.dot(dq_c_inv[1:4], dq_t[1:4])
    dq_e[1:4] = dq_c_inv[0]*dq_t[1:4] + dq_t[0]*dq_c_inv[1:4] + np.cross(dq_c_inv[1:4], dq_t[1:4])
    # Dual part multiplication
    dq_e[4] = dq_c_inv[0]*dq_t[4] - np.dot(dq_c_inv[1:4], dq_t[5:8]) + dq_c_inv[4]*dq_t[0] - np.dot(dq_c_inv[5:8], dq_t[1:4])
    dq_e[5:8] = (dq_c_inv[0]*dq_t[5:8] + dq_t[4]*dq_c_inv[1:4] + np.cross(dq_c_inv[1:4], dq_t[5:8]) +
                 dq_c_inv[4]*dq_t[1:4] + dq_t[0]*dq_c_inv[5:8] + np.cross(dq_c_inv[5:8], dq_t[1:4]))

    # 3. Shortest Path & Dual Logarithm
    if dq_e[0] < 0.0:
        dq_e = -dq_e

    v_r_norm = np.linalg.norm(dq_e[1:4])
    axis_ang = np.empty(6)

    if v_r_norm < 1e-6:
        # Pure translation singularity
        axis_ang[0:3] = 2.0 * dq_e[5:8]
        axis_ang[3:6] = 0.0
    else:
        # Standard combined spatial twist
        theta = 2.0 * np.arctan2(v_r_norm, dq_e[0])
        n = dq_e[1:4] / v_r_norm
        d = -2.0 * dq_e[4] / v_r_norm
        m = (dq_e[5:8] - n * (d / 2.0) * dq_e[0]) / v_r_norm
        
        omega = theta * n          # omega
        v = d * n + theta * m  # v

    # 4. Apply Gain
    axis_ang = np.concatenate((v, omega))
    v_cmd = k_diag * axis_ang
    
    # 5. Check Arrival
    arrived = np.sum(np.abs(axis_ang)) < threshold
    
    return v_cmd, arrived


### Method D: ScLERP Derivative

This method implements the analytical derivation.


$$
\begin{aligned}
&\begin{gathered}
\boldsymbol{D}(s)=\boldsymbol{D}_1\left(\boldsymbol{D}_1^{-1} \boldsymbol{D}_2\right)^s=\boldsymbol{D}_1 \boldsymbol{D}_{12}^s=\boldsymbol{D}_1\left(\cos \frac{s \Phi}{2}+\boldsymbol{L} \sin \frac{s \Phi}{2}\right) \begin{array}{cc}
\Phi=\theta+\epsilon d & \begin{array}{l}
z=a+\epsilon b\left(\epsilon^2=0\right) \\
\sin (z)=\sin (a)+\epsilon b \cos (a) \\
\boldsymbol{L}=\widehat{\boldsymbol{s}}+\epsilon \boldsymbol{m}
\end{array} \quad \begin{array}{c}
\cos (z)=\cos (a)-\epsilon b \sin (a)
\end{array} \quad(*) \\
\quad\left(\boldsymbol{D}_1^*=\boldsymbol{D}_1^{-1}\right)
\end{array} \\
\quad=\boldsymbol{D}_1\left(\left(\cos \frac{s \theta}{2}, \hat{\boldsymbol{s}} \sin \frac{s \theta}{2}\right)+\epsilon\left(-\frac{s d}{2} \sin \frac{s \theta}{2}, \frac{s d}{2} \cos \frac{s \theta}{2} \hat{\boldsymbol{s}}+\sin \frac{s \theta}{2} \boldsymbol{m}\right)\right) \\
\dot{\boldsymbol{D}}(s)=\boldsymbol{D}_1 \frac{\dot{\boldsymbol{D}_{12}^s}}{2}=\boldsymbol{D}_1\left(-\frac{\dot{s} \Phi}{2} \sin \frac{s \Phi}{2}+\boldsymbol{L} \frac{\dot{s} \Phi}{2} \cos \frac{s \Phi}{2}\right)=\boldsymbol{D}_1\left(-\frac{\dot{s}(\theta+\epsilon d)}{2} \sin \frac{s(\theta+\epsilon d)}{2}+(\hat{\boldsymbol{s}}+\epsilon \boldsymbol{m}) \frac{\dot{s}(\theta+\epsilon d)}{2} \cos \frac{s(\theta+\epsilon d)}{2}\right) \\
=\ldots
\end{gathered}\\
&\text { We can simplify this by using (*) to compute } \dot{\boldsymbol{D}}(s) \text { and substitute into following to find } \boldsymbol{V}_b \text { or } \boldsymbol{V}_s \text {. }\\
&\begin{aligned}
& \overline{\mathcal{V}}_b=2 \boldsymbol{D}^{-1}(s) \dot{\boldsymbol{D}}(s) \xrightarrow{\overline{\mathcal{V}}_b=\left(0, \omega_b\right)+\epsilon\left(0, \boldsymbol{v}_b\right)} \boldsymbol{v}_b=\left[\begin{array}{l}
\omega_b \\
\boldsymbol{v}_b
\end{array}\right] \\
& \overline{\boldsymbol{V}}_s=2 \dot{\boldsymbol{D}} \boldsymbol{D}^{-1}=\boldsymbol{D} \overline{\boldsymbol{V}}_b \boldsymbol{D}^{-1} \xrightarrow{\overline{\boldsymbol{V}}_s=\left(0, \omega_s\right)+\epsilon\left(0, \boldsymbol{v}_s\right)} \boldsymbol{v}_s=\left[\begin{array}{c}
\omega_s \\
\boldsymbol{v}_s
\end{array}\right]
\end{aligned}
\end{aligned}
$$


In [ ]:
def twist_from_sclerp_derivative(dq_c: np.ndarray, dq_t: np.ndarray, s: float = 0.0, s_dot: float = 1.0):
    """
    Computes body twist V_b = [v, omega] using the ScLERP time derivative.
    """
    # 1. Compute relative error D12 = dq_c^{-1} * dq_t
    dq_c_inv = np.concatenate((pr.q_conj(dq_c[:4]), pr.q_conj(dq_c[4:])))
    D12 = pt.concatenate_dual_quaternions(dq_c_inv, dq_t)
    
    # Enforce shortest path
    if D12[0] < 0.0:
        D12 = -D12
        
    # 2. Extract Screw Parameters
    q, n, h, theta = pt.screw_parameters_from_dual_quaternion(D12)
    
    if np.isinf(h): # Pure translation
        d, theta = theta, 0.0
        m = np.zeros(3)
    else:
        d = h * theta
        m = np.cross(q, n)
        
    # 3. Formulate Dual Scalars
    a, b = s * theta / 2, s * d / 2
    cos_sPhi_2 = np.array([np.cos(a), 0, 0, 0, -b * np.sin(a), 0, 0, 0])
    sin_sPhi_2 = np.array([np.sin(a), 0, 0, 0, b * np.cos(a), 0, 0, 0])
    s_dot_Phi_2 = np.array([s_dot * theta / 2, 0, 0, 0, s_dot * d / 2, 0, 0, 0])
    
    # 4. Formulate L = n + e * m
    L = np.concatenate(([0], n, [0], m))
    
    # 5. Compute d/dt (D12^s)
    term1 = -pt.concatenate_dual_quaternions(s_dot_Phi_2, sin_sPhi_2)
    term2_scalar = pt.concatenate_dual_quaternions(s_dot_Phi_2, cos_sPhi_2)
    term2 = pt.concatenate_dual_quaternions(L, term2_scalar)
    d_dt_D12_s = term1 + term2
    
    # 6. D_dot(s) = D1 * d/dt(D12^s)
    D_dot_s = pt.concatenate_dual_quaternions(dq_c, d_dt_D12_s)
    
    # 7. D(s) = D1 * D12^s
    D12_s = cos_sPhi_2 + pt.concatenate_dual_quaternions(L, sin_sPhi_2)
    D_s = pt.concatenate_dual_quaternions(dq_c, D12_s)
    
    # 8. Body Twist: V_b = 2 * D(s)^-1 * D_dot(s)
    D_s_inv = np.concatenate((pr.q_conj(D_s[:4]), pr.q_conj(D_s[4:])))
    V_b_dq = 2 * pt.concatenate_dual_quaternions(D_s_inv, D_dot_s)
    
    # 9. Extract and flip to [v, omega] to match p_servo_dq exactly
    omega_b = V_b_dq[1:4]
    v_b = V_b_dq[5:8]
    
    return np.concatenate((v_b, omega_b))


## 3. Verification Benchmark

This script confirms that all three derivations yield identically scaled target twists to guarantee kinematic consistency.


In [89]:

if __name__ == "__main__":
    # 1. Setup Arbitrary Test Data
    T_c = pt.transform_from(
        R=pr.matrix_from_axis_angle([0, 0, 1, np.pi/4]), 
        p=[1.0, 2.0, 3.0]
    )
    T_t = pt.transform_from(
        R=pr.matrix_from_axis_angle([1, 0, 0, -np.pi/3]), 
        p=[1.5, 2.5, 2.8]
    )

    k = 1.0
    threshold = 1e-3

    # 2. Verify Output Parity
    dq_c = pt.dual_quaternion_from_transform(T_c)
    dq_t = pt.dual_quaternion_from_transform(T_t)

    v_mat, arr_mat = p_servo_matrix(T_c, T_t, k, threshold)
    v_dq, arr_dq = p_servo_dq(dq_c, dq_t, k, threshold)
    v_dq_fast, arr_dq_fast = p_servo_dq_fast(dq_c, dq_t, k * np.ones(6), threshold)
    v_sclerp = twist_from_sclerp_derivative(dq_c, dq_t, s=0.0, s_dot=1.0)   
    v_sclerp_mid = twist_from_sclerp_derivative(dq_c, dq_t, s=0.5, s_dot=1.0)
    
    # Map twists back to transformation matrices
    def to_mr_format(twist):
        return np.concatenate([twist[3:], twist[:3]])

    T_from_mat = mr.MatrixExp6(mr.VecTose3(to_mr_format(v_mat)))
    T_from_dq = mr.MatrixExp6(mr.VecTose3(to_mr_format(v_dq)))

    print("--- Output Verification ---")
    print(f"Matrix Twist: \n{np.round(T_from_mat, 5)}")
    print(f"DQ Twist:     \n{np.round(T_from_dq, 5)}")
    print("Is the resulting physical pose identical?")
    print(np.allclose(T_from_mat, T_from_dq, atol=1e-4))

    print("--- Output Verification ---")
    print(f"Matrix Twist: {np.round(v_mat, 5)}")
    print(f"DQ Twist:     {np.round(v_dq, 5)}")
    print(f"Fast Twist:   {np.round(v_dq_fast, 5)}")
    print(f"ScLERP Twist: {np.round(v_sclerp, 5)}")
    print(f"ScLERP Mid Twist: {np.round(v_sclerp_mid, 5)}")

    print(f"Match (Matrix-DQ):        {np.allclose(v_mat, v_dq)}")
    print(f"Match (Matrix-Fast):      {np.allclose(v_mat, v_dq_fast)}\n")
    print(f"Match (Matrix-ScLERP):    {np.allclose(v_mat, v_sclerp)}")

    # 3. Run Benchmark
    iterations = 10_000

    time_mat = timeit.timeit(lambda: p_servo_matrix(T_c, T_t, k, threshold), number=iterations)
    
    time_dq = timeit.timeit(lambda: p_servo_dq(dq_c, dq_t, k, threshold), number=iterations)
    time_dq_fast = timeit.timeit(lambda: p_servo_dq_fast(dq_c, dq_t, k * np.ones(6), threshold), number=iterations)
    
    print(f"--- Benchmark Results ({iterations} iterations) ---")
    print(f"Matrix (modern_robotics): {time_mat:.4f} seconds")
    print(f"Dual Quat (pytransform3d): {time_dq:.4f} seconds")
    print(f"Dual Quat Fast (optimized): {time_dq_fast:.4f} seconds")
    print(f"Speed Multiplier:          {time_mat / time_dq:.2f}x faster")
    print(f"Fast Multiplier:           {time_mat / time_dq_fast:.2f}x faster")

--- Output Verification ---
Matrix Twist: 
[[ 0.70711  0.35355  0.61237  0.70711]
 [-0.70711  0.35355  0.61237  0.     ]
 [-0.      -0.86603  0.5     -0.2    ]
 [ 0.       0.       0.       1.     ]]
DQ Twist:     
[[ 0.70711  0.35355  0.61237  0.70711]
 [-0.70711  0.35355  0.61237 -0.     ]
 [-0.      -0.86603  0.5     -0.2    ]
 [ 0.       0.       0.       1.     ]]
Is the resulting physical pose identical?
True
--- Output Verification ---
Matrix Twist: [ 0.69523  0.33076  0.00752 -0.99082  0.41041 -0.71086]
DQ Twist:     [ 0.69523  0.33076  0.00752 -0.99082  0.41041 -0.71086]
Fast Twist:   [ 0.69523  0.33076  0.00752 -0.99082  0.41041 -0.71086]
ScLERP Twist: [ 1.08068  0.51414  0.01169 -1.54015  0.63795 -1.10497]
ScLERP Mid Twist: [ 1.54061  0.32329  0.34193 -1.53962  0.63773 -1.10458]
Match (Matrix-DQ):        True
Match (Matrix-Fast):      True

Match (Matrix-ScLERP):    False
--- Benchmark Results (10000 iterations) ---
Matrix (modern_robotics): 1.1995 seconds
Dual Quat (pytrans

Here is exactly why the outputs do not match.

### **1. The Gain Mismatch ($k$ vs $\dot{s}$)**

Look closely at the magnitudes of the outputs at $s=0.0$:

* **Matrix / DQ Twist:** `[ 0.69523, 0.33076, 0.00752 ... ]`
* **ScLERP Twist ($s=0$):** `[ 1.08068, 0.51414, 0.01169 ... ]`

If you divide the Matrix twist by the ScLERP twist ($0.69523 / 1.08068$), the ratio is exactly **$0.6433$**. This can be compensated by $\dot{s}$. The desired twist can be scaled withouth changing the direction that end-effector follows. 


### **2. The Numerical Drift (Why $s=0.5$ changed)**

It can be seen that the linear velocity evaluated at $s=0.5$ changed drastically from $s=0.0$. **Mathematically, this should be impossible.** If we take the analytical derivation and expand it, the parameter $s$ completely cancels out:


$$\mathcal{V}_b = 2 \boldsymbol{D}^{-1}(s) \dot{\boldsymbol{D}}(s)$$


Substitute $\boldsymbol{D}(s) = \boldsymbol{D}_1 \boldsymbol{D}_{12}^s$ and $\dot{\boldsymbol{D}}(s) = \boldsymbol{D}_1 \dot{\boldsymbol{D}}_{12}^s$:


$$\mathcal{V}_b = 2 (\boldsymbol{D}_1 \boldsymbol{D}_{12}^s)^{-1} (\boldsymbol{D}_1 \dot{\boldsymbol{D}}_{12}^s)$$

$$\mathcal{V}_b = 2 (\boldsymbol{D}_{12}^s)^{-1} \boldsymbol{D}_1^{-1} \boldsymbol{D}_1 \dot{\boldsymbol{D}}_{12}^s$$


Because $\boldsymbol{D}_1^{-1} \boldsymbol{D}_1 = 1$, the global start pose cancels out. If you push the trigonometric limits through the rest of the equation, the $(\boldsymbol{D}_{12}^s)^{-1}$ cancels with the time derivative, leaving only a single, perfectly constant term:


$$\mathcal{V}_b = \dot{s} \Phi \boldsymbol{L}$$

**Why did Python fail?**
Dual scalars ($\Phi$) and sine/cosine expansions commute mathematically. But when you force dual scalars into $8 \times 1$ arrays and pass them through `pytransform3d.concatenate_dual_quaternions`, the library applies standard rigid-body spatial multiplications (which assume cross-products). This breaks the commutative property of the dual scalars at $s \neq 0$, causing massive numerical drift.


In [94]:
v_dq_fast / v_sclerp

array([0.64332918, 0.64332918, 0.64332918, 0.64332918, 0.64332918,
       0.64332918])